# Grounded local generation

The local model only improves wording. The planner and sensor adapter keep control of every subject, duration, break, and action.

In [ ]:
import copy
import json
from pathlib import Path

from src.deterministic_response import deterministic_grounded_response
from src.generation_validator import validate_generated_response
from src.grounding_context import build_grounding_context

root = Path.cwd()
inputs = json.loads((root / 'data' / 'processed' / 'generation_demo_inputs.json').read_text(encoding='utf-8'))
outputs = json.loads((root / 'data' / 'processed' / 'generation_demo_outputs.json').read_text(encoding='utf-8'))
demo = inputs[0]
print(demo['case_id'])

## Deterministic input

This plan is already complete before language generation starts.

In [ ]:
plan = demo['plan']
print([(item['subject'], item['allocated_study_minutes']) for item in plan['scheduled_subjects']])
print([(item['order'], item['session_type'], item['duration_minutes']) for item in plan['sessions']])

## Minimized grounding context

Only the final plan, relevant record summaries, optional sensor result, and original request are supplied.

In [ ]:
context = build_grounding_context(
    demo['original_user_input'],
    demo['plan'],
    retrieved_record_summaries=demo['retrieved_record_summaries'],
    sensor_result=demo['sensor_result'],
)
print(context.keys())
print(len(context['retrieved_record_summaries']))

## Generated response and validation

The committed response came from the real local pass or its validated fallback mode.

In [ ]:
generated = outputs[0]
validation = validate_generated_response(generated['response'], context)
print(generated['generation_mode'])
print(generated['response']['summary'])
print(validation.valid, validation.errors)
assert validation.valid

## Rejected alteration

A model response that changes an allocated duration is rejected.

In [ ]:
altered = copy.deepcopy(generated['response'])
altered['allocation_explanations'][0]['allocated_minutes'] += 5
rejected = validate_generated_response(altered, context)
print(rejected.valid, rejected.errors)
assert not rejected.valid

## Stable fallback and value comparison

The fallback remains useful when Ollama is unavailable or both model attempts fail.

In [ ]:
fallback_a = deterministic_grounded_response(context)
fallback_b = deterministic_grounded_response(context)
assert fallback_a == fallback_b
before_allocations = [(item['subject'], item['allocated_minutes']) for item in context['final_plan']['subject_allocations']]
after_allocations = [(item.subject, item.allocated_minutes) for item in fallback_a.allocation_explanations]
before_sessions = [item['order'] for item in context['final_plan']['sessions']]
after_sessions = [item.session_order for item in fallback_a.session_messages]
print({'allocations_unchanged': before_allocations == after_allocations, 'session_orders_unchanged': before_sessions == after_sessions})
assert before_allocations == after_allocations
assert before_sessions == after_sessions